# Oliver's ANSYS UPF pool — reproducible analysis

Re-derives, from the delivered files themselves, the findings written up in
[`OLIVER_MODEL_NOTES.md`](OLIVER_MODEL_NOTES.md). Nothing here is hard-coded
from those notes: every number and every table below is read out of the
sources at run time, so re-running this against a newer delivery shows what
changed.

**The files are not in the repository** — they are another group's code and
~26 MB of binaries. Point `POOL` at wherever they are unpacked.

Covers:

1. inventory of the delivery
2. the `usermat` signature difference between ANSYS releases (the thing that
   blocks building on v222)
3. the constitutive call sites — which material is live, which is disabled
4. the ecology in `usercm.inc` / `Ussfin`
5. the NEM derivative operator
6. a side-by-side against this repo's own model

In [1]:
from pathlib import Path
import re, textwrap

# --- point these at your unpacked copies -------------------------------------
POOL = Path("/tmp/genoliver")                     # .../Nishioka_Hoechel/ANSYS-Pool
REPO = Path("..").resolve()                       # this repository
DECK = None    # optional: path to ds.dat from the .wbpz, or leave None
# -----------------------------------------------------------------------------

assert POOL.exists(), f"POOL not found: {POOL}"
print("pool :", POOL)
print("repo :", REPO)

pool : /tmp/genoliver
repo : /home/user/pde-fem-biofilm


## 1. What was delivered

In [2]:
rows=[]
for f in sorted(POOL.iterdir()):
    if f.is_file():
        rows.append((f.name, f.stat().st_size))
w=max(len(n) for n,_ in rows)
src=[(n,s) for n,s in rows if n.lower().endswith((".f",".inc",".h"))]
print(f"{len(rows)} files, {sum(s for _,s in rows)/1e6:.1f} MB total\n")
print("Fortran / headers:")
for n,s in src:
    print(f"  {n:<{w}} {s/1024:8.1f} kB")

32 files, 6.0 MB total

Fortran / headers:
  AGPhaseViskoP21V07.f                    161.4 kB
  AGStressP21V07.f                         14.0 kB
  AceGenElastoAirV08.f                      5.5 kB
  AceGenNeoHookV02.f                       16.5 kB
  AceGenNeoHookV03.f                       15.9 kB
  AceGenNeoHookV04.f                       15.6 kB
  MySubroutines_userData_V04.F             24.0 kB
  NEM_UserData_P21_V05.F                   70.4 kB
  USolBeg_P21-V21_Conection_Test.F         39.0 kB
  Usermat_P21-V21_Conection_Test.F         32.9 kB
  Ussfin_P21-V21_Conection_Test.F         162.0 kB
  sms.h                                    10.0 kB
  usercm.inc                                8.3 kB
  usercm_P21-V21_Conection_Test.inc         8.3 kB
  userdata_P21-V21_Conection_Test.f         5.5 kB


## 2. The `usermat` signature — why v222 is not a drop-in

ANSYS changes this argument list between releases, and the repo's own
`README.md` flags it as the first thing to re-check when moving versions.
Here it is counted rather than assumed.

In [3]:
def usermat_args(path):
    """Parse the usermat subroutine argument list out of a Fortran source."""
    src = Path(path).read_text(errors="replace")
    m = re.search(r"subroutine\s+usermat\s*\((.*?)\)", src, re.S | re.I)
    if not m:
        return None
    body = m.group(1)
    body = re.sub(r"\n\s*[c*!].*", "", body)      # comment lines
    body = re.sub(r"\n\s*&", "", body)            # continuations
    return [a.strip() for a in body.split(",") if a.strip()]

theirs = usermat_args(POOL / "Usermat_P21-V21_Conection_Test.F")
ours   = usermat_args(REPO / "ansys_usermat" / "usermat_biofilm.f")

print(f"Oliver (ANSYS 2024 R2) : {len(theirs)} args")
print(f"this repo (v222)       : {len(ours)} args")
print()
print("tail after cutFactor")
cut = lambda a: a[a.index("cutFactor")+1:]
print("  2024 R2 :", ", ".join(cut(theirs)))
print("  v222    :", ", ".join(cut(ours)))
print()
only_t = [a for a in theirs if a not in ours]
only_o = [a for a in ours if a not in theirs]
print("only in 2024 R2 :", only_t)
print("only in v222    :", only_o)

Oliver (ANSYS 2024 R2) : 41 args
this repo (v222)       : 42 args

tail after cutFactor
  2024 R2 : pVolDer, hrmflg, var3, var4, var5, var6, var7
  v222    : var1, var2, var3, var4, var5, var6, var7, var8

only in 2024 R2 : ['pVolDer', 'hrmflg']
only in v222    : ['var1', 'var2', 'var8']


The two reserved slots became named arguments and one was dropped, so the
same source cannot compile against both. `pVolDer` is an *array* where `var1`
is a scalar, which is why this has to be fixed deliberately rather than by
renaming — see [`apdl/V222_PORT_INSTRUCTIONS.md`](apdl/V222_PORT_INSTRUCTIONS.md).

## 3. Which material is actually live

In [4]:
um = (POOL / "Usermat_P21-V21_Conection_Test.F").read_text(errors="replace")

calls = []
for i, line in enumerate(um.splitlines(), 1):
    m = re.match(r"\s*(!?)\s*CALL\s+(AceGen\w+|AG\w+)", line, re.I)
    if m:
        calls.append((i, "disabled" if m.group(1) == "!" else "LIVE", m.group(2)))

for ln, state, name in calls:
    print(f"  line {ln:>5}  {state:<8}  {name}")

print()
print("Named in the disabled block:",
      re.search(r"!-+\s*(Matmodell \w+) Start", um).group(1))

  line   554  LIVE      AceGenNeoHookV04
  line   562  disabled  AceGenElastoAirV08
  line   592  disabled  AGStressP21V07

Named in the disabled block: Matmodell Tobi


Only one call is live, and it is not a stand-in: read its arguments against
its own signature.

In [5]:
sig = re.search(r"SUBROUTINE\s+AceGenNeoHookV04\((.*?)\)",
                (POOL / "AceGenNeoHookV04.f").read_text(errors="replace"), re.S | re.I)
formal = [a.strip() for a in re.sub(r"\n\s*&", "", sig.group(1)).split(",")]

call = re.search(r"CALL\s+AceGenNeoHookV04\((.*?)\)", um, re.S | re.I)
actual = [a.strip() for a in re.sub(r"\n\s*&", "", call.group(1)).split(",")]

print(f"{'formal':<16} <- actual")
for f, a in zip(formal, actual):
    print(f"  {f:<14} <- {a}")

formal           <- actual
  v              <- Vdp_AceGen
  mDefGrad       <- defGrad
  vCauchy        <- stress
  mTangCC        <- dsdePl
  sYoung         <- sGdp_YoungBio
  sYoungL        <- sGdp_YoungVoid
  sNu            <- sGdp_PoissonBio
  sNuL           <- sGdp_PoissonVoid
  sBiofilm       <- Sdp_sumBio
  sAlpha         <- Sdp_sumLocal
  sElasticWork   <- sedEl
  sID            <- ID


Note `sAlpha <- Sdp_sumLocal`. **That is not the growth α** — it is a local
biofilm average. Confirm from where those are assigned:

In [6]:
for line in um.splitlines():
    if re.match(r"\s*Sdp_sum(Bio|Local)\s*=", line):
        print(line.rstrip())
print()
print("declaration comment:")
for line in um.splitlines():
    if "Summe biofilm" in line:
        print(" ", line.strip())

      Sdp_sumBio = Sdp_bio1_n + Sdp_bio1_n
      Sdp_sumLocal = (Sdp_locbio1_n + Sdp_locbio2_n)/2

declaration comment:
  &, Sdp_sumBio, Sdp_sumLocal !Summe biofilm/local Biofilm


The first line adds `bio1` to itself where the second averages `locbio1` and
`locbio2` — flagged to Oliver as a possible typo rather than treated as a
conclusion, since we cannot run the build.

## 4. The ecology: two species, two nutrients

In [7]:
inc = (POOL / "usercm.inc").read_text(errors="replace")
groups = {
    "species / nutrient start":        r"sGdp_(?:Bio|Nut)\dstart",
    "max growth (species x nutrient)": r"sGdp_MaxGrowth\d\d",
    "half velocity":                   r"sGdp_HalfVelo\d\d",
    "pairwise interaction":            r"sGdp_Interaction\d\d",
    "diffusion":                       r"sGdp_Diff\d",
}
for label, pat in groups.items():
    found = sorted(set(re.findall(pat, inc)))
    print(f"{label:<34}", ", ".join(found) if found else "(none)")

species / nutrient start           sGdp_Bio1start, sGdp_Bio2start, sGdp_Nut1start, sGdp_Nut2start
max growth (species x nutrient)    sGdp_MaxGrowth11, sGdp_MaxGrowth12, sGdp_MaxGrowth21, sGdp_MaxGrowth22
half velocity                      sGdp_HalfVelo11, sGdp_HalfVelo12, sGdp_HalfVelo21, sGdp_HalfVelo22
pairwise interaction               sGdp_Interaction12, sGdp_Interaction21
diffusion                          sGdp_Diff1, sGdp_Diff2


In [8]:
uf = (POOL / "Ussfin_P21-V21_Conection_Test.F").read_text(errors="replace")
i = uf.index("!Growth function Bio 1")
print(textwrap.dedent(uf[i:i+700]))

!Growth function Bio 1
      GrowthBio1 = 0.0D0
      GrowthBio1 = SQRT(Sdp_LapBio1**2) *
     &(
     &    ( (sGdp_MaxGrowth11 + sGdp_Interaction12 * vGdp_Bio2_n(ID)) 
     &      * vGdp_Nut1_n(ID) )
     &    / (sGdp_HalfVelo11 + vGdp_Nut1_n(ID))
     &  + ( (sGdp_MaxGrowth21 + sGdp_Interaction12 * vGdp_Bio2_n(ID)) 
     &      * vGdp_Nut2_n(ID) )
     &    / (sGdp_HalfVelo21 + vGdp_Nut2_n(ID))
     & )

!Growth function Bio 2
      GrowthBio2 = 0.0D0
      GrowthBio2 = SQRT(Sdp_LapBio2**2) *
     &(
     &    ( (sGdp_MaxGrowth12 + sGdp_Interaction21 * vGdp_Bio1_n(ID)) 
     &      * vGdp_Nut1_n(ID) )
     &    / (sGdp_HalfVelo12 + vGdp_Nut1_n(ID))
     &  + ( (sGdp_MaxGrowth22 + sGd


Monod kinetics per nutrient, with the *other* species shifting the maximum
growth rate — the "novel interaction scheme" of the published paper (Klempt,
Geisler, Soleimani et al., *Archive of Applied Mechanics* **96**, 164 (2026)) —
and `|∇²Bio|` localising growth at the front.

This repo carries the same interaction as a full matrix `A`, calibrated by
TMCMC, at five species: 

In [9]:
nsp = (REPO / "JAXFEM" / "hamilton_ode_jax_nsp.py").read_text()
print("hamilton_ode_jax_nsp.py — parameter count vs species count")
m = re.search(r"def count_params\(.*?\n(?:.*?\n)*?\s*return .*", nsp)
print(m.group(0) if m else "(not found)")

import sys
sys.path.insert(0, str(REPO / "JAXFEM"))
try:
    import hamilton_ode_jax_nsp as hn
    for n in (2, 4, 5):
        print(f"  n_sp={n:>2}  ->  {hn.count_params(n)} parameters")
except Exception as e:
    print("(jax not installed here:", e, ")")

hamilton_ode_jax_nsp.py — parameter count vs species count
def count_params(n_sp, k_hill_free=False):
    """Number of parameters for N species."""
    n_A = n_sp * (n_sp + 1) // 2
    return n_A + n_sp + (1 if k_hill_free else 0)


  n_sp= 2  ->  5 parameters
  n_sp= 4  ->  14 parameters
  n_sp= 5  ->  20 parameters


## 5. The NEM operator is a weighted least-squares derivative

In [10]:
nem = (POOL / "NEM_UserData_P21_V05.F").read_text(errors="replace")

print("Gaussian kernel:")
for line in nem.splitlines():
    if "mW(ii,ii)=EXP" in line.replace(" ", "") or "mWNode(ii,ii)=EXP" in line.replace(" ", ""):
        print("  ", line.strip())

print("\nmoment-matrix inversion:")
for line in nem.splitlines():
    if "InversGauss" in line:
        print("  ", line.strip())

print("\nhow the rows are consumed (fixes what each row means):")
asm = nem[nem.index("SUBROUTINE AssembleSparse"):]
for line in asm.splitlines()[:130]:
    if re.search(r"mD\(\d", line):
        print("  ", line.strip())

Gaussian kernel:
   mW(ii,ii)=EXP(-0.5*((4.0D0*SQRT((vdx(ii)**2.0D0)+(vdy(ii)**2.0D0)
   mWNode(ii,ii)=EXP(-0.5D0*((4.0D0*SQRT((vdxNode(ii)**2.0D0)

moment-matrix inversion:
   EXTERNAL :: erhandler,InversGauss,dlange
   CALL InversGauss(mResult3,9)
   CALL InversGauss(mResult3Node,4)

how the rows are consumed (fixes what each row means):
   DOUBLE PRECISION :: mD(9,sGi_NeighCnt), mDNode(4,sGi_NeighCnt+1)
   &       ( mD(1,ii) + mD(2,ii) + mD(3,ii) )
   &       ( mD(1,ii) + mD(2,ii) + mD(3,ii) )
   Vdp_Dx_T(1) = Vdp_Dx_T(1) - mD(7,ii)
   Vdp_Dx_T(1+ii) =  mD(7,ii)
   Vdp_Dy_T(1) = Vdp_Dy_T(1) - mD(8,ii)
   Vdp_Dy_T(1+ii) =  mD(8,ii)
   Vdp_Dz_T(1) = Vdp_Dz_T(1) - mD(9,ii)
   Vdp_Dz_T(1+ii) =  mD(9,ii)


Rows 1–3 sum to the Laplacian, rows 7–9 are the first derivatives. The
self-coefficient is set to minus the sum of the neighbour coefficients, so the
operator annihilates constants — the zeroth-order consistency condition for
this class of scheme.

## 6. Where the two codebases meet

In [11]:
summary = [
    ("field solve",        "NEM (weighted least squares) + PARDISO, in ANSYS",
                           "finite differences in JAX (JAXFEM/)"),
    ("species",            "2",                       "5 fixed, general n in *_nsp"),
    ("interaction",        "Interaction12 / Interaction21",
                           "calibrated matrix A (TMCMC)"),
    ("bounding [0,1]",     "penalty",                 "log barrier"),
    ("elastic law",        "AceGenNeoHookV04 (bio/void blend)",
                           "Mooney-Rivlin + D1"),
    ("viscous law",        "glass model only (disabled)",
                           "backward-Euler Fv, verified 0 ULP vs Abaqus"),
    ("growth Fg=(1+a)I",   "absent",                  "present"),
]
w0 = max(len(a) for a, _, _ in summary)
w1 = max(len(b) for _, b, _ in summary)
print(f"{'':<{w0}}  {'Oliver':<{w1}}  this repo")
print("-" * (w0 + w1 + 30))
for a, b, c in summary:
    print(f"{a:<{w0}}  {b:<{w1}}  {c}")

                  Oliver                                            this repo
----------------------------------------------------------------------------------------------
field solve       NEM (weighted least squares) + PARDISO, in ANSYS  finite differences in JAX (JAXFEM/)
species           2                                                 5 fixed, general n in *_nsp
interaction       Interaction12 / Interaction21                     calibrated matrix A (TMCMC)
bounding [0,1]    penalty                                           log barrier
elastic law       AceGenNeoHookV04 (bio/void blend)                 Mooney-Rivlin + D1
viscous law       glass model only (disabled)                       backward-Euler Fv, verified 0 ULP vs Abaqus
growth Fg=(1+a)I  absent                                            present


**Complementary rather than overlapping.** They have the field machinery and a
biofilm elastic law at n=2; this repo has the calibrated n-species ecology, the
verified viscous law, and the growth kinematics. The open decisions — which
ANSYS release to target, who computes the field, whether the AceGen notebook can
be shared — are in the draft to Oliver and in `OLIVER_MODEL_NOTES.md`.